In [8]:
import pandas as pd

# 1. 파일 불러오기
file_path = 'filtered_data.csv'
df = pd.read_csv(file_path)

# 2. '카테고리'가 '대출상품 조회'인 데이터만 필터링
loan_df = df[df['카테고리'] == '대출상품 조회'].copy()

# 3. 본문 파싱 함수 정의
def parse_loan_content(text):
    headers = ['기본정보', '상품정보', '상품요건', '지원대상요건', '기타 상품정보']
    end_marker = '목록'
    
    indices = []
    for h in headers:
        idx = text.find(h)
        if idx != -1:
            indices.append((h, idx))
    
    indices.sort(key=lambda x: x[1])
    
    end_idx = text.find(end_marker)
    if end_idx == -1:
        end_idx = len(text)
    indices.append(('End', end_idx))
    
    parsed_data = {h: '' for h in headers}
    
    for i in range(len(indices) - 1):
        current_header, current_idx = indices[i]
        next_header, next_idx = indices[i+1]
        
        content_start = current_idx + len(current_header)
        content = text[content_start:next_idx].strip()
        
        if current_header in parsed_data:
            parsed_data[current_header] = content
            
    return parsed_data

# 4. 데이터프레임에 파싱 적용
parsed_rows = []
for index, row in loan_df.iterrows():
    parsed = parse_loan_content(row['본문'])
    # Add other columns to the dictionary for easy DataFrame creation
    parsed['카테고리'] = row['카테고리']
    parsed['공고제목'] = row['공고제목']
    parsed_rows.append(parsed)

# 5. 결과 데이터프레임 생성
result_df = pd.DataFrame(parsed_rows)

# 6. 컬럼 순서 설정 (사용자 요청: 카테고리, 공고제목, 기본정보, 상품정보, 상품요건, 지원대상요건, 기타 상품정보)
cols_order = ['카테고리', '공고제목', '기본정보', '상품정보', '상품요건', '지원대상요건', '기타 상품정보']
result_df = result_df[cols_order]

# 7. 결과 저장
output_filename = 'parsed_loan_products_final.csv'
result_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"File saved to {output_filename}")
print(result_df.head())

File saved to parsed_loan_products_final.csv
      카테고리                            공고제목  \
0  대출상품 조회                 소상공인 비즈플러스 카드보증   
1  대출상품 조회    2025년 중구 소상공인 경영안정자금 지원 특례보증   
2  대출상품 조회  2025 강원특별자치도 소상공인 경영안정자금 협약보증(   
3  대출상품 조회           2025 원주시 출연 소상공인 협약보증   
4  대출상품 조회       2025 속초시 출연 소상공인 협약보증(변경)   

                                                기본정보  \
0        금융상품명 소상공인 비즈플러스 카드보증 금리(%) - 최대한도(만원) 1000   
1  금융상품명 2025년 중구 소상공인 경영안정자금 지원 특례보증 금리(%) 은행별 상...   
2  금융상품명 2025 강원특별자치도 소상공인 경영안정자금 협약보증(하반기) 금리(%)...   
3  금융상품명 2025 원주시 출연 소상공인 협약보증 금리(%) 3 최대한도(만원) 5000   
4  금융상품명 2025 속초시 출연 소상공인 협약보증(변경) 금리(%) - 최대한도(만...   

                                                상품정보  \
0                   대상 소상공인 용도 - 취급기관 기업은행 대출기간(년) 1   
1               대상 소상공인 용도 운영 취급기관 아이엠뱅크 대출기간(년) 2,5   
2  대상 소상공인, 기업 용도 - 취급기관 농협, 신한, 국민, 하나, 우리 대출기간(...   
3  대상 소상공인 용도 - 취급기관 NH농협은행, 국민은행, 기업은행, SH수협은행, ...   
4        대상 소상공인 용도 - 취급기관 재단과 협약 체결한 금융회사 대출기간(년) -   

    